# Lab 01-01 — Fixed vs recursive text splitting: where the cuts land

**Track 01 · Chunking** — the first decision in every RAG pipeline is *how do I cut a long document into chunks?* This lab runs the two classic answers head to head on the same three public-domain Project Gutenberg novels — with the same `chunk_size`/`chunk_overlap` — and then inspects *where* the cuts land, because that is what determines whether embeddings see clean text at every chunk boundary.

This notebook is **self-contained**: it imports LangChain splitters directly — no repo component library. Every block of the pipeline is built right here: the Gutenberg loader (a plain file read plus a marker strip), the fixed splitter, and the recursive splitter all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

```
raw text  ->  inline Gutenberg stripper  ->  CharacterTextSplitter(separator="")   ->  boundary analysis
                                          ->  RecursiveCharacterTextSplitter        ->  boundary analysis
Data  : Data/corpus/gutenberg/ (pride-and-prejudice, moby-dick, a-tale-of-two-cities)
Budget: chunk_size=500, chunk_overlap=50
```

**FIXED splitting** is a pure character counter: it cuts at exactly `chunk_size` characters, no matter where that lands. Words and sentences get torn in half; the chunk count is purely a function of text length. (Note: `CharacterTextSplitter` defaults to a `\n\n` separator, which would quietly respect paragraphs — passing `separator=""` turns it into the blind character counter that "fixed splitting" really means.)

**RECURSIVE splitting** climbs a ladder of separators — paragraphs (`\n\n`), newlines (`\n`), spaces — and only falls back to characters when nothing else fits. Chunks end on paragraph/word boundaries instead of mid-word.

Why it matters: embeddings are trained on whole words and sentences. A chunk that starts or ends mid-word feeds garbage tokens at exactly the boundary points where neighbouring chunks meet, so retrieval quality degrades where structure matters most. The lab measures both chunk populations (count, average/min/max length) and the boundary quality directly: how many fixed cuts tear a word in half, and whether recursive cuts land on chapter headings.


## Setup

One prerequisite must hold before this notebook will run:

- **gutenberg corpus on disk** — `Data/corpus/gutenberg/` with `pride-and-prejudice.txt`, `moby-dick.txt`, `a-tale-of-two-cities.txt` (public-domain novels, already fetched by the repo's manifest-verified fetchers).

No repo imports are needed: everything this notebook uses comes from `langchain-core` and `langchain-text-splitters`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q langchain-core langchain-text-splitters


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import re
from pathlib import Path

# LangChain splitters — the only libraries this notebook needs. Nothing is
# imported from the repo's src/ component library.
from langchain_core.documents import Document  # noqa: E402
from langchain_text_splitters import CharacterTextSplitter  # noqa: E402
from langchain_text_splitters import RecursiveCharacterTextSplitter  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything the lab measures is driven by these named constants. `CHUNK_SIZE` is the target length of every chunk, `CHUNK_OVERLAP` is how many characters bleed into the next chunk (so context is never lost exactly at a boundary), and `PREVIEW` controls how much text is shown on each side of a cut in the boundary analysis. `DOC_PATHS` is the corpus: three public-domain Gutenberg novels, whose license preamble/footer the loader strips before splitting.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the comparison
# --------------------------------------------------------------------------
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
PREVIEW = 100  # max characters shown on each side of a cut
DOC_PATHS = [
    Path("Data/corpus/gutenberg/pride-and-prejudice.txt"),
    Path("Data/corpus/gutenberg/moby-dick.txt"),
    Path("Data/corpus/gutenberg/a-tale-of-two-cities.txt"),
]


## 2. Load — Gutenberg books, boilerplate stripped inline

The repo's `GutenbergLoader` strips everything between the `*** START OF THE PROJECT GUTENBERG EBOOK` and `*** END OF THE PROJECT GUTENBERG EBOOK` markers and sets `metadata["source"]` to the book path. That boilerplate is noise for chunking — the license preamble would otherwise be split and embedded like real content. The inline version below does exactly the same two things: read the file as UTF-8 text, slice out the book between the markers, and wrap it as one `Document` per book.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — Gutenberg books via the inline boilerplate stripper
# --------------------------------------------------------------------------
START_MARKER = "*** START OF THE PROJECT GUTENBERG EBOOK"
END_MARKER = "*** END OF THE PROJECT GUTENBERG EBOOK"


def strip_gutenberg_boilerplate(text: str) -> str:
    """Slice from just after the START marker to just before the END marker.

    If either marker is missing the text is returned unchanged (defensive:
    some mirrors drop the footer), and the result is stripped of leading/
    trailing whitespace — same semantics as the repo's GutenbergLoader.
    """
    start = text.find(START_MARKER)
    end = text.find(END_MARKER)
    if start == -1 or end == -1 or end < start:
        return text.strip()
    return text[start + len(START_MARKER):end].strip()


def load_docs(paths: list[Path]) -> list[Document]:
    """Load each book as one Document (boilerplate stripped, source tagged)."""
    docs = []
    for path in paths:
        text = strip_gutenberg_boilerplate(path.read_text(encoding="utf-8"))
        docs.append(
            Document(page_content=text, metadata={"source": str(path)})
        )
    return docs


## 3. Experiment — split both ways, measure where the cuts land

`run_experiment` runs the head-to-head: both splitters get the same three books and the same `chunk_size=500, chunk_overlap=50`, then the resulting chunk populations are compared on length stats (count, average, min, max) and on boundary quality. Boundary quality is the teaching point, measured three ways:

* `cut_stats` — of all in-document cuts, how many land right at a chapter heading (the next chunk opens with a `Chapter` line);
* `find_midword_cut` — the first in-document cut where FIXED tears a word in half, reconstructed from the letter runs on both sides of the cut (the overlap repeat is stripped from the next chunk's preview so the reader sees the continuation of the torn word);
* `find_chapter_boundary` — the first in-document cut where RECURSIVE lands on a chapter heading (the overlap repeat stripped from the tail, so the reader sees where the previous chunk truly ended).

Gutenberg novels are plain text, so structure shows up as `CHAPTER 1`/`CHAPTER I` headings, not markdown `#` headings — the `CHAPTER_RE` regex recognizes both.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — helpers for stats and boundary-quality detection
# --------------------------------------------------------------------------
def chunk_length_stats(chunks: list[Document]) -> tuple[int, float, int, int]:
    """Return (count, avg, min, max) chunk length in characters."""
    lengths = [len(c.page_content) for c in chunks]
    count = len(lengths)
    if count == 0:
        return 0, 0.0, 0, 0
    return count, sum(lengths) / count, min(lengths), max(lengths)


def escape(s: str) -> str:
    """Make newlines visible so cut positions are obvious in the output."""
    return s.replace("\n", "\\n")


def torn_word(tail: str, head: str) -> str:
    """Reconstruct the word torn across a cut from the letter runs on both sides."""
    trailing = []
    for ch in reversed(tail):
        if ch.isalnum():
            trailing.append(ch)
        else:
            break
    leading = []
    for ch in head:
        if ch.isalnum():
            leading.append(ch)
        else:
            break
    return "".join(reversed(trailing)) + "".join(leading)


def overlap_suffix(prev: str, next_: str) -> str:
    """Longest suffix of ``prev`` that is also a prefix of ``next_`` (the overlap repeat)."""
    for n in range(min(len(prev), len(next_)), 0, -1):
        if prev[-n:] == next_[:n]:
            return prev[-n:]
    return ""


# Gutenberg novels are plain text: structure shows up as "CHAPTER 1"/"CHAPTER I"
# headings, not markdown "#" headings.
CHAPTER_RE = re.compile(r"(?i)^chapter\s+[0-9ivxlcdm.]+")


def cut_stats(chunks: list[Document]) -> tuple[int, int]:
    """Return (in-document cuts, cuts where the next chunk opens a chapter heading)."""
    total = 0
    chapter_aligned = 0
    for i in range(len(chunks) - 1):
        if chunks[i].metadata.get("source") != chunks[i + 1].metadata.get("source"):
            continue
        total += 1
        if CHAPTER_RE.match(chunks[i + 1].page_content.lstrip()):
            chapter_aligned += 1
    return total, chapter_aligned


def find_midword_cut(chunks: list[Document]) -> tuple[int, str, str, str] | None:
    """First in-document cut where fixed splitting tears a word in half.

    Returns (index, tail_preview, fresh_head_preview, torn_word). The next
    chunk's preview has the ``CHUNK_OVERLAP`` repeat stripped so the reader
    sees the continuation of the torn word, not the duplicated tail.
    """
    for i in range(len(chunks) - 1):
        if chunks[i].metadata.get("source") != chunks[i + 1].metadata.get("source"):
            continue
        tail = chunks[i].page_content
        head = chunks[i + 1].page_content
        overlap = tail[-CHUNK_OVERLAP:]
        fresh = head[len(overlap):] if head.startswith(overlap) else head
        if tail and fresh and tail[-1].isalnum() and fresh[0].isalnum():
            return i, tail[-PREVIEW:], fresh[:PREVIEW], torn_word(tail, fresh)
    return None


def find_chapter_boundary(chunks: list[Document]) -> tuple[int, str, str] | None:
    """First in-document cut where the next chunk opens a chapter heading.

    Returns (index, tail_preview, head_preview). The tail has the overlap
    repeat stripped so the reader sees where the previous chunk truly ended.
    """
    for i in range(len(chunks) - 1):
        if chunks[i].metadata.get("source") != chunks[i + 1].metadata.get("source"):
            continue
        head = chunks[i + 1].page_content
        if CHAPTER_RE.match(head.lstrip()):
            tail = chunks[i].page_content
            overlap = overlap_suffix(tail, head)
            if overlap:
                tail = tail[: -len(overlap)]
            return i, tail[-PREVIEW:], head[:PREVIEW]
    return None


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — run the head-to-head
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    docs = load_docs(DOC_PATHS)

    # --- Split with both splitters (same chunk_size/chunk_overlap) -------
    fixed_chunks = CharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separator=""
    ).split_documents(docs)
    recursive_chunks = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    ).split_documents(docs)

    # --- Compare lengths ------------------------------------------------
    f_stats = chunk_length_stats(fixed_chunks)
    r_stats = chunk_length_stats(recursive_chunks)

    # --- Boundary quality -----------------------------------------------
    f_cuts, f_aligned = cut_stats(fixed_chunks)
    r_cuts, r_aligned = cut_stats(recursive_chunks)
    midword = find_midword_cut(fixed_chunks)
    boundary = find_chapter_boundary(recursive_chunks)

    return {
        "docs": docs,
        "fixed_chunks": fixed_chunks,
        "recursive_chunks": recursive_chunks,
        "f_stats": f_stats,
        "r_stats": r_stats,
        "f_cuts": f_cuts,
        "f_aligned": f_aligned,
        "r_cuts": r_cuts,
        "r_aligned": r_aligned,
        "midword": midword,
        "boundary": boundary,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from five angles: the loaded books with their sizes; the chunk counts each splitter produced; the length stats table; the boundary-quality report (chapter-aligned cuts, the torn-word reconstruction from the FIXED split, the chapter boundary from the RECURSIVE split); and the takeaway.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    docs = exp["docs"]
    fixed_chunks = exp["fixed_chunks"]
    recursive_chunks = exp["recursive_chunks"]

    print("=" * 66)
    print("Lab 01 — fixed vs recursive text splitting")
    print(f"chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")
    print("=" * 66)
    print(f"\n[1] Loaded {len(docs)} document(s):")
    for doc in docs:
        print(f"    {Path(doc.metadata['source']).name:<22} {len(doc.page_content):>5} chars")

    print("\n[2] Split with both splitters (same chunk_size/chunk_overlap):")
    print(f"    CharacterTextSplitter(separator='') : {len(fixed_chunks):>2} chunks")
    print(f"    DocumentProcessor (recursive)        : {len(recursive_chunks):>2} chunks")

    f_count, f_avg, f_min, f_max = exp["f_stats"]
    r_count, r_avg, r_min, r_max = exp["r_stats"]
    print("\n[3] Chunk length stats (characters):")
    print(f"    {'':<12}{'chunks':>7}{'avg':>9}{'min':>7}{'max':>7}")
    print(f"    {'fixed':<12}{f_count:>7d}{f_avg:>9.1f}{f_min:>7d}{f_max:>7d}")
    print(f"    {'recursive':<12}{r_count:>7d}{r_avg:>9.1f}{r_min:>7d}{r_max:>7d}")

    print("\n[4] Where do the cuts land?")
    print(f"    cuts landing at a chapter heading (next chunk opens a 'Chapter'): "
          f"fixed {exp['f_aligned']}/{exp['f_cuts']}, recursive {exp['r_aligned']}/{exp['r_cuts']}")

    midword = exp["midword"]
    if midword is not None:
        i, tail, fresh, word = midword
        src = Path(fixed_chunks[i].metadata["source"]).name
        print(f"\n    FIXED cuts at exactly {CHUNK_SIZE} chars, mid-word:")
        print(f"      {src} chunk {i} ends    : ...{escape(tail)}")
        print(f"      {src} chunk {i + 1} (overlap stripped) starts: {escape(fresh)}...")
        print(f"      -> the word '{word}' is torn in half across chunks {i} and {i + 1}")
    else:
        print("\n    FIXED: no mid-word cut found (text too short or boundaries aligned).")

    boundary = exp["boundary"]
    if boundary is not None:
        j, tail, head = boundary
        src = Path(recursive_chunks[j].metadata["source"]).name
        print("\n    RECURSIVE climbs the separator ladder and lands on a chapter boundary:")
        print(f"      {src} chunk {j} ends  : ...{escape(tail)}")
        print(f"      {src} chunk {j + 1} starts: {escape(head)}...")
        print(f"      -> cut lands at a paragraph boundary; chunk {j + 1} opens a clean chapter")
    else:
        print("\n    RECURSIVE: no chapter boundary found (whole doc fits in one chunk).")

    print("\n[5] Takeaway")
    print("    Fixed splitting counts characters; recursive splitting counts")
    print("    structure. Same 500/50 budget, but recursive chunks keep words")
    print("    and paragraphs intact, so embeddings see clean text at every")
    print("    chunk boundary.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: all three books loaded with the Gutenberg boilerplate actually stripped; both splitters honour the 500-char budget (no chunk over `CHUNK_SIZE`); the FIXED split demonstrably tears a word in half (the teaching point); and the RECURSIVE split demonstrably lands on a chapter heading. Every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append((f"{len(DOC_PATHS)} Gutenberg books loaded",
                   len(exp["docs"]) == len(DOC_PATHS)))
    checks.append(("Gutenberg boilerplate stripped from every book",
                   all(START_MARKER not in d.page_content and END_MARKER not in d.page_content
                       for d in exp["docs"])))
    checks.append(("fixed chunks honour the 500-char budget (max <= CHUNK_SIZE)",
                   exp["f_stats"][3] <= CHUNK_SIZE))
    checks.append(("recursive chunks honour the 500-char budget (max <= CHUNK_SIZE)",
                   exp["r_stats"][3] <= CHUNK_SIZE))
    checks.append(("FIXED tears a word in half somewhere (mid-word cut found)",
                   exp["midword"] is not None and bool(exp["midword"][3])))
    checks.append(("RECURSIVE lands on a chapter heading somewhere",
                   exp["boundary"] is not None))
    checks.append(("FIXED cut stats reported (chapter-aligned/total is a fraction)",
                   exp["f_cuts"] > 0 and exp["f_aligned"] <= exp["f_cuts"]))
    checks.append(("RECURSIVE cut stats reported (chapter-aligned/total is a fraction)",
                   exp["r_cuts"] > 0 and exp["r_aligned"] <= exp["r_cuts"]))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few seconds: three plain-text reads and two character-level splits of ~2MB of public-domain prose — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The head-to-head on identical input and budget: chunk counts, length stats, and — the point of the lab — where each splitter's cuts land. FIXED tears words in half at exactly 500 chars; RECURSIVE climbs its separator ladder and lands on chapter boundaries.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the gutenberg files are intact.


In [ ]:
verify_gate(exp)
